In [3]:
import cv2
import os
import pandas as pd
import numpy as np

# --- CONFIGURATION ---
DATASET_DIR = "extra1"          # Input Folder containing original images
OUTPUT_LABELS = "train_labels_extra1.csv"
OUTPUT_IMG_DIR = "extra_Labeled_Images_Additional1" # Output Folder for highlighted images
IMG_WIDTH, IMG_HEIGHT = 800, 600
ROWS, COLS = 8, 8
CELL_W, CELL_H = 100, 75

# Colors for visualization (BGR)
COLORS = {
    0: None,            # Background
    1: (0, 255, 0),     # Ball (Green)
    2: (0, 0, 255),     # Bat (Red)
    3: (255, 0, 0)      # Stump (Blue)
}

CLASS_NAMES = {0: "Background", 1: "Ball", 2: "Bat", 3: "Stump"}

# State Variables
current_labels = {} # Dictionary to store {cell_id: label_code}
image_data = []     # List to store final rows for CSV
current_img = None
current_img_name = ""

def draw_grid_and_labels(img):
    """Overlays grid and colored labels on the image."""
    display_img = img.copy()
    
    # Draw Grid
    for i in range(1, COLS):
        cv2.line(display_img, (i * CELL_W, 0), (i * CELL_W, IMG_HEIGHT), (255, 255, 0), 1)
    for i in range(1, ROWS):
        cv2.line(display_img, (0, i * CELL_H), (IMG_WIDTH, i * CELL_H), (255, 255, 0), 1)
        
    # Draw Active Labels
    for cell_id, label in current_labels.items():
        if label == 0: continue
        
        # Calculate cell coordinates
        r = cell_id // COLS
        c = cell_id % COLS
        x1, y1 = c * CELL_W, r * CELL_H
        x2, y2 = x1 + CELL_W, y1 + CELL_H
        
        # Draw filled rectangle with transparency
        color = COLORS[label]
        sub_img = display_img[y1:y2, x1:x2]
        colored_rect = np.zeros(sub_img.shape, dtype=np.uint8)
        colored_rect[:] = color
        res = cv2.addWeighted(sub_img, 0.6, colored_rect, 0.4, 1.0)
        display_img[y1:y2, x1:x2] = res
        
        # Optional: Add text label for clarity
        # cv2.putText(display_img, str(label), (x1+5, y1+20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1)
        
    return display_img

def mouse_callback(event, x, y, flags, param):
    """Handles mouse clicks to toggle labels."""
    global current_labels
    
    if event == cv2.EVENT_LBUTTONDOWN:
        # Determine Cell ID
        c = x // CELL_W
        r = y // CELL_H
        cell_id = r * COLS + c
        
        # Cycle Label: 0 -> 1 -> 2 -> 3 -> 0
        current_val = current_labels.get(cell_id, 0)
        new_val = (current_val + 1) % 4
        current_labels[cell_id] = new_val
        
        print(f"Cell {cell_id} set to {CLASS_NAMES[new_val]}")

def save_csv():
    """Helper to save current data to CSV."""
    df = pd.DataFrame(image_data, columns=['ImageName', 'CellID', 'y'])
    df.to_csv(OUTPUT_LABELS, index=False)
    print(f"Labels saved to {OUTPUT_LABELS}")

def main():
    global current_img, current_labels, current_img_name
    
    # 1. Setup Directories
    if not os.path.exists(DATASET_DIR):
        print(f"Error: Input directory '{DATASET_DIR}' not found.")
        return

    if not os.path.exists(OUTPUT_IMG_DIR):
        os.makedirs(OUTPUT_IMG_DIR)
        print(f"Created output directory: {OUTPUT_IMG_DIR}")

    # 2. Load Images
    images = [f for f in os.listdir(DATASET_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    images.sort()
    
    if not images:
        print("No images found in dataset directory!")
        return

    # Check for existing labels to resume (optional logic placeholder)
    if os.path.exists(OUTPUT_LABELS):
        print("Found existing labels file. Appending new data...")
    
    cv2.namedWindow("Labeling Tool")
    cv2.setMouseCallback("Labeling Tool", mouse_callback)
    
    print("\nControls:\n  [Click] Toggle Label\n  [n] Save & Next Image\n  [q] Save & Quit")
    
    for img_name in images:
        current_img_name = img_name
        img_path = os.path.join(DATASET_DIR, img_name)
        img = cv2.imread(img_path)
        
        if img is None: continue
        if img.shape[:2] != (IMG_HEIGHT, IMG_WIDTH):
            img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))
            
        current_img = img
        current_labels = {i: 0 for i in range(64)} # Reset labels for new image
        
        while True:
            # Render
            display_img = draw_grid_and_labels(current_img)
            cv2.imshow("Labeling Tool", display_img)
            
            key = cv2.waitKey(1) & 0xFF
            
            # 'n' for Next
            if key == ord('n'):
                # 1. Save CSV Data
                for cell_id, label in current_labels.items():
                    image_data.append([current_img_name, cell_id, label])
                
                # 2. Save Highlighted Image
                save_path = os.path.join(OUTPUT_IMG_DIR, img_name)
                cv2.imwrite(save_path, display_img)
                
                print(f"Saved data and image for {img_name}")
                break
                
            # 'q' for Quit
            elif key == ord('q'):
                print("Quitting...")
                save_csv()
                return

    # Save Final CSV if loop finishes naturally
    save_csv()
    print("All images processed.")
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()


Controls:
  [Click] Toggle Label
  [n] Save & Next Image
  [q] Save & Quit
Cell 3 set to Ball
Cell 3 set to Bat
Cell 11 set to Ball
Cell 11 set to Bat
Cell 19 set to Ball
Cell 19 set to Bat
Cell 20 set to Ball
Cell 20 set to Bat
Cell 28 set to Ball
Cell 28 set to Bat
Cell 27 set to Ball
Cell 27 set to Bat
Cell 35 set to Ball
Cell 35 set to Bat
Cell 36 set to Ball
Cell 36 set to Bat
Cell 44 set to Ball
Cell 44 set to Bat
Cell 43 set to Ball
Cell 43 set to Bat
Cell 51 set to Ball
Cell 51 set to Bat
Cell 52 set to Ball
Cell 52 set to Bat
Cell 60 set to Ball
Cell 60 set to Bat
Cell 59 set to Ball
Cell 59 set to Bat
Saved data and image for Image500.jpg
Cell 4 set to Ball
Cell 4 set to Bat
Cell 12 set to Ball
Cell 12 set to Bat
Cell 20 set to Ball
Cell 20 set to Bat
Cell 19 set to Ball
Cell 19 set to Bat
Cell 27 set to Ball
Cell 27 set to Bat
Cell 28 set to Ball
Cell 28 set to Bat
Cell 36 set to Ball
Cell 36 set to Bat
Cell 35 set to Ball
Cell 35 set to Bat
Cell 43 set to Ball
Cell 43 set 